# AMIHUD_20 非流动性因子（预热集：2004–2005；训练集：2006–2017）

本 Notebook 只负责因子生产，并且只使用信号日及以前的数据：

- 日度价格冲击：`abs(复权收益率) / 成交额（元） × 1e8`；
- 固定最近20个市场交易日，至少15个有效成交日；
- 使用2004–2005年行情预热滚动窗口与上市天数；
- 排除沪深以外股票、当日ST、上市不足120个交易日的新股；
- 仅在2006-01-04至2017-12-29训练区间执行 `log1p`、MAD去极值、规模中性化和Z-score；
- 预热数据不进入训练因子文件，也不参与训练期横截面统计。


## 1. 依赖与参数

默认值固定为2004–2005预热期和2006–2017正式训练期。环境变量仅用于冒烟测试，不改变正式默认参数。


In [ ]:
from pathlib import Path
from IPython.display import display
import importlib.util
import os
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Back_test').exists() and (p / 'Factor').exists()), Path.cwd())
DATA_PATH = PROJECT_DIR / 'Data' / 'stocks_daily_04-25.parquet'
NOTEBOOK_DIR = PROJECT_DIR / 'Factor'

WARMUP_START_DATE = os.getenv('AMIHUD_WARMUP_START_DATE', '2017-11-01')
START_DATE = os.getenv('AMIHUD_START_DATE', '2018-01-02')
END_DATE = os.getenv('AMIHUD_END_DATE', '2021-12-31')
OUTPUT_DIR = Path(os.getenv(
    'AMIHUD_OUTPUT_DIR',
    str(NOTEBOOK_DIR / 'amihud_20_output_20180102_20211231'),
))

ROLLING_WINDOW = 20
MIN_VALID_DAYS = 15
MIN_LISTING_DAYS = 120
AMOUNT_SCALE = 100_000_000.0
MAD_SCALE = 3.0
MIN_CROSS_SECTION = 100
EXCLUDE_ST = True
EXCHANGES = ('SH', 'SZ')

# 因子生产默认不自动回测；正式回测交给通用回测模板。
RUN_BACKTEST = os.getenv('AMIHUD_RUN_BACKTEST', '0') == '1'
HORIZON = 20
EXECUTION_LAG = 1
REBALANCE_STEP = 20
GROUPS = 10
COST_BPS = 20
MAX_MISSING_RETURN_RATIO = 0.20

WARMUP_START_TS = pd.Timestamp(WARMUP_START_DATE)
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE)
WARMUP_START_INT = int(WARMUP_START_TS.strftime('%Y%m%d'))
END_INT = int(END_TS.strftime('%Y%m%d'))
DATE_TAG = f'{START_TS:%Y%m%d}_{END_TS:%Y%m%d}'

assert WARMUP_START_TS < START_TS <= END_TS
assert DATA_PATH.exists(), f'找不到行情文件：{DATA_PATH}'
assert 1 <= MIN_VALID_DAYS <= ROLLING_WINDOW

print('行情：', DATA_PATH)
print('预热集：', WARMUP_START_DATE, '至', (START_TS - pd.Timedelta(days=1)).date())
print('训练集：', START_DATE, '至', END_DATE)
print('输出：', OUTPUT_DIR)
print('自动回测：', RUN_BACKTEST)


## 2. 分批读取预热集与训练集行情

只读取计算需要的字段。成交额由千元转换为元；市场日历剔除横截面不足100只股票的异常日期。不会读取训练期结束后的验证集或测试集数据。


In [ ]:
RAW_COLUMNS = [
    '股票代码', '股票名称', '交易日', '收盘价',
    '成交额（千元）', '总市值（万元）', '复权因子'
]


def load_market_window(path, start_int, end_int, batch_size=300_000):
    parquet_file = pq.ParquetFile(path)
    selected = []
    scanned = 0
    for batch in parquet_file.iter_batches(
        batch_size=batch_size, columns=RAW_COLUMNS, use_threads=True
    ):
        table = pa.Table.from_batches([batch])
        dates = table['交易日']
        mask = pc.and_(
            pc.greater_equal(dates, pa.scalar(start_int)),
            pc.less_equal(dates, pa.scalar(end_int)),
        )
        filtered = table.filter(mask)
        if filtered.num_rows:
            selected.append(filtered)
        scanned += table.num_rows
    assert selected, '预热及训练区间内没有数据'
    result = pa.concat_tables(selected).to_pandas()
    print(f'扫描 {scanned:,} 行，预热及训练区间保留 {len(result):,} 行')
    return result


market = load_market_window(DATA_PATH, WARMUP_START_INT, END_INT)
market = market.rename(columns={
    '股票代码': 'code',
    '股票名称': 'name',
    '交易日': 'date_int',
    '收盘价': 'close',
    '成交额（千元）': 'amount_thousand',
    '总市值（万元）': 'total_market_cap',
    '复权因子': 'adjustment_factor',
})
exchange_suffixes = tuple(f'.{exchange}' for exchange in EXCHANGES)
market = market[market['code'].astype(str).str.endswith(exchange_suffixes)].copy()
market['date'] = pd.to_datetime(market['date_int'], format='%Y%m%d', errors='coerce')
market = market.drop(columns='date_int')
for column in ['close', 'amount_thousand', 'total_market_cap', 'adjustment_factor']:
    market[column] = pd.to_numeric(market[column], errors='coerce')

date_counts = market.groupby('date')['code'].nunique()
market_dates = pd.Index(date_counts[date_counts >= MIN_CROSS_SECTION].index.sort_values())
market = market[market['date'].isin(market_dates)].copy()
date_to_index = pd.Series(np.arange(len(market_dates)), index=market_dates)
market['date_index'] = market['date'].map(date_to_index).astype('int32')
market = market.sort_values(['code', 'date'], kind='stable').reset_index(drop=True)

assert not market.duplicated(['date', 'code']).any(), '存在重复股票日期'
assert market['date'].min() >= WARMUP_START_TS
assert market['date'].max() <= END_TS
assert (market['date'] < START_TS).any(), '没有读到正式训练期之前的预热数据'

print('实际读取日期：', market['date'].min().date(), '至', market['date'].max().date())
print('预热行情行数：', f"{(market['date'] < START_TS).sum():,}")
print('训练行情行数：', f"{market['date'].between(START_TS, END_TS).sum():,}")
print('市场交易日：', len(market_dates))
print('沪深记录：', f'{len(market):,}')
print('股票数量：', f"{market['code'].nunique():,}")
display(market.head())


## 3. 计算日度价格冲击和20日原始因子

收益必须与上一市场交易日连续。滚动窗口固定为最近20个市场日，不允许因为停牌或缺失而向更早日期扩张；窗口内至少需要15个有效成交日。2004–2005年行情先参与收益、上市天数和滚动窗口预热，计算后才截取正式训练期。


In [ ]:
valid_close = (
    np.isfinite(market['close']) & (market['close'] > 0)
    & np.isfinite(market['adjustment_factor']) & (market['adjustment_factor'] > 0)
)
valid_amount = np.isfinite(market['amount_thousand']) & (market['amount_thousand'] > 0)
valid_market_cap = np.isfinite(market['total_market_cap']) & (market['total_market_cap'] > 0)
market['adjusted_close'] = (market['close'] * market['adjustment_factor']).where(valid_close)
market['amount_yuan'] = (market['amount_thousand'] * 1000.0).where(valid_amount)
market['log_market_cap'] = np.log(market['total_market_cap'].where(valid_market_cap))

grouped = market.groupby('code', sort=False)
previous_close = grouped['adjusted_close'].shift(1)
previous_date_index = grouped['date_index'].shift(1)
consecutive_return = (market['date_index'] - previous_date_index).eq(1)
adjusted_return = market['adjusted_close'] / previous_close - 1.0
valid_return = consecutive_return & np.isfinite(adjusted_return) & (adjusted_return > -1.0)
market['adjusted_return'] = adjusted_return.where(valid_return)
market['daily_amihud'] = (
    market['adjusted_return'].abs() / market['amount_yuan'] * AMOUNT_SCALE
).where(valid_return & valid_amount)

is_st = (
    market['name'].fillna('').str.upper().str.contains('ST', regex=False)
    if EXCLUDE_ST else pd.Series(False, index=market.index)
)
first_valid_index = market['date_index'].where(valid_close & valid_amount).groupby(
    market['code'], sort=False
).transform('min')
market['listing_age'] = market['date_index'] - first_valid_index

# 在预热起点已上市的股票，其真实上市日早于可见数据边界；到2006年训练期
# 已至少拥有约两年历史，可视为满足120个交易日要求。
seasoned = first_valid_index.eq(0) | market['listing_age'].ge(MIN_LISTING_DAYS)

market['amihud_valid_days'] = grouped['daily_amihud'].transform(
    lambda values: values.rolling(ROLLING_WINDOW, min_periods=1).count()
)
market['amihud_20_raw'] = grouped['daily_amihud'].transform(
    lambda values: values.rolling(ROLLING_WINDOW, min_periods=MIN_VALID_DAYS).mean()
)
window_start_index = grouped['date_index'].shift(ROLLING_WINDOW - 1)
fixed_window = (market['date_index'] - window_start_index).eq(ROLLING_WINDOW - 1)
market.loc[~fixed_window, 'amihud_20_raw'] = np.nan
market['amihud_20_log'] = np.log1p(market['amihud_20_raw'])

target_mask = market['date'].between(START_TS, END_TS)
eligible_signal = (
    target_mask
    & market['amihud_20_log'].notna()
    & valid_market_cap & seasoned & (~is_st)
)
market.loc[~eligible_signal, 'amihud_20_log'] = np.nan

# 验证训练首日：所有满足固定窗口、有效日数、上市期和可投资条件的记录
# 都必须已经生成原始因子，不能再因数据集起点发生冷启动。
assert target_mask.any(), '正式训练区间内没有行情数据'
first_training_date = market.loc[target_mask, 'date'].min()
first_day_mask = market['date'].eq(first_training_date)
first_day_eligible = (
    first_day_mask & fixed_window
    & market['amihud_valid_days'].ge(MIN_VALID_DAYS)
    & valid_market_cap & seasoned & (~is_st)
)
assert market.loc[first_day_eligible, 'amihud_20_log'].notna().all(), (
    '训练首日仍存在由滚动边界造成的意外缺失'
)

warmup_diagnostics = pd.Series({
    '正式训练首日': first_training_date,
    '首日行情记录数': int(first_day_mask.sum()),
    '首日固定窗口且可投资记录数': int(first_day_eligible.sum()),
    '首日有效原始因子数': int(
        market.loc[first_day_mask, 'amihud_20_log'].notna().sum()
    ),
    '首日窗口不足或不连续数': int(
        (first_day_mask & (
            market['amihud_valid_days'].lt(MIN_VALID_DAYS) | ~fixed_window
        )).sum()
    ),
})
display(warmup_diagnostics.to_frame('值'))

quality_summary = pd.Series({
    '预热及训练记录数': len(market),
    '有效日度Amihud': int(market['daily_amihud'].notna().sum()),
    '训练期有效20日原始因子': int(market['amihud_20_log'].notna().sum()),
    'ST记录': int(is_st.sum()),
    '未满上市期记录': int((~seasoned).sum()),
})
display(quality_summary.to_frame('数量'))

# 从这里开始只保留正式训练期；预热数据不参与横截面处理和最终输出。
market = market.loc[target_mask].copy().reset_index(drop=True)
assert market['date'].between(START_TS, END_TS).all()

display(market.loc[market['amihud_20_log'].notna(), [
    'date', 'code', 'adjusted_return', 'amount_yuan',
    'daily_amihud', 'amihud_valid_days', 'amihud_20_raw', 'amihud_20_log'
]].head())


## 4. 每日MAD、规模中性化和标准化

只在正式训练期每日执行横截面处理。最终 `amihud_20_factor` 越大，表示在相同规模下价格冲击越强、流动性越差。


In [ ]:
date_group = market.groupby('date', sort=False)['amihud_20_log']
daily_median = date_group.transform('median')
absolute_deviation = (market['amihud_20_log'] - daily_median).abs()
daily_mad = absolute_deviation.groupby(market['date'], sort=False).transform('median')
market['amihud_20_winsor'] = market['amihud_20_log'].clip(
    daily_median - MAD_SCALE * daily_mad,
    daily_median + MAD_SCALE * daily_mad,
).where(daily_mad > 0, market['amihud_20_log'])

eligible = market['amihud_20_winsor'].notna() & market['log_market_cap'].notna()
x = market['log_market_cap'].where(eligible)
y = market['amihud_20_winsor'].where(eligible)
key = market['date']
daily_n = eligible.groupby(key, sort=False).transform('sum')
x_mean = x.groupby(key, sort=False).transform('mean')
y_mean = y.groupby(key, sort=False).transform('mean')
dx = x - x_mean
dy = y - y_mean
cov_xy = (dx * dy).groupby(key, sort=False).transform('sum')
var_x = dx.pow(2).groupby(key, sort=False).transform('sum')
valid_regression = (daily_n >= MIN_CROSS_SECTION) & (var_x > 0)
size_beta = (cov_xy / var_x).where(valid_regression)
market['amihud_20_neutral'] = (dy - size_beta * dx).where(valid_regression)

neutral_group = market.groupby('date', sort=False)['amihud_20_neutral']
neutral_mean = neutral_group.transform('mean')
neutral_std = neutral_group.transform('std')
market['amihud_20_factor'] = (
    (market['amihud_20_neutral'] - neutral_mean)
    / neutral_std.where(neutral_std > 0)
)

residual = market['amihud_20_neutral']
residual_centered = residual - residual.groupby(key, sort=False).transform('mean')
residual_cov = (dx * residual_centered).groupby(key, sort=False).transform('sum')
residual_var = residual_centered.pow(2).groupby(key, sort=False).transform('sum')
market['_neutral_size_corr'] = residual_cov / np.sqrt(var_x * residual_var)
market['_size_beta'] = size_beta

factor_coverage = market[market['date'].between(START_TS, END_TS)].groupby('date').agg(
    market_rows=('code', 'size'),
    valid_daily_amihud=('daily_amihud', 'count'),
    raw_factor=('amihud_20_raw', 'count'),
    valid_factor=('amihud_20_factor', 'count'),
    median_valid_days=('amihud_valid_days', 'median'),
    size_beta=('_size_beta', 'first'),
    neutralized_size_correlation=('_neutral_size_corr', 'first'),
)
factor_coverage['coverage'] = factor_coverage['valid_factor'] / factor_coverage['market_rows']
valid_corr = factor_coverage['neutralized_size_correlation'].dropna()
assert not valid_corr.empty
assert valid_corr.abs().max() < 1e-7, '规模中性化检查失败'
display(factor_coverage.describe().T)

## 5. 保存训练因子和覆盖率审计

输出严格限制在2006–2017训练区间，预热记录不会进入因子文件。


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
factor_columns = [
    'date', 'code', 'name', 'adjusted_return', 'amount_yuan',
    'daily_amihud', 'amihud_valid_days', 'listing_age',
    'amihud_20_raw', 'amihud_20_log', 'amihud_20_winsor',
    'amihud_20_neutral', 'amihud_20_factor'
]
factor_panel = market.loc[
    market['amihud_20_factor'].notna(), factor_columns
].copy()
factor_panel['asof_date'] = factor_panel['date']

assert factor_panel['date'].between(START_TS, END_TS).all()
assert factor_panel['date'].min() == market['date'].min()
assert not factor_panel.duplicated(['date', 'code']).any()
assert np.isfinite(factor_panel['amihud_20_factor']).all()
assert (factor_panel['amihud_valid_days'] >= MIN_VALID_DAYS).all()

factor_path = OUTPUT_DIR / f'amihud_20_factor_{DATE_TAG}.parquet'
coverage_path = OUTPUT_DIR / 'factor_coverage.csv'
factor_panel.to_parquet(factor_path, index=False, compression='zstd')
factor_coverage.to_csv(coverage_path, index_label='date')

summary = pd.Series({
    '因子记录数': len(factor_panel),
    '股票数量': factor_panel['code'].nunique(),
    '有效首日': factor_panel['date'].min(),
    '有效末日': factor_panel['date'].max(),
    '平均覆盖率': factor_coverage.loc[factor_coverage['valid_factor'] > 0, 'coverage'].mean(),
    '中性化后最大绝对规模相关': valid_corr.abs().max(),
    '输出大小MB': factor_path.stat().st_size / 1024 / 1024,
})
display(summary.to_frame('值'))
print('因子文件：', factor_path)
print('覆盖率审计：', coverage_path)


## 6. 可选：严格接入通用回测

默认不自动运行。因子已经完成横截面预处理，因此回测使用 `preprocessed=True`，不重复去极值或中性化。正式研究建议使用 `Back_test/通用因子回测模板.ipynb`。


In [ ]:
if RUN_BACKTEST:
    backtest_dir = PROJECT_DIR / 'Back_test'
    backtest_engine_path = backtest_dir / 'backtest.py'
    assert backtest_engine_path.is_file(), f'回测引擎不存在：{backtest_engine_path}'

    engine_spec = importlib.util.spec_from_file_location(
        'strict_factor_backtest_engine_amihud', backtest_engine_path
    )
    assert engine_spec is not None and engine_spec.loader is not None
    backtest_engine = importlib.util.module_from_spec(engine_spec)
    sys.modules[engine_spec.name] = backtest_engine
    engine_spec.loader.exec_module(backtest_engine)

    BacktestConfig = backtest_engine.BacktestConfig
    DataPaths = backtest_engine.DataPaths
    FactorSpec = backtest_engine.FactorSpec
    StrictFactorBacktester = backtest_engine.StrictFactorBacktester

    backtest_output = (
        backtest_dir / 'outputs' / 'amihud_20' / f'{HORIZON}d_{DATE_TAG}'
    )
    paths = DataPaths(
        factor_dir=factor_path,
        market_parquet=DATA_PATH,
        market_cache_dir=backtest_dir / '.cache' / 'market_data',
        output_dir=backtest_output,
    )
    factor = FactorSpec(
        name='amihud_20', column='amihud_20_factor',
        direction='positive', preprocessed=True,
    )
    config = BacktestConfig(
        start_date=START_DATE, end_date=END_DATE,
        horizon=HORIZON, execution_lag=EXECUTION_LAG,
        rebalance_step=REBALANCE_STEP, groups=GROUPS,
        cost_bps=COST_BPS, min_stocks=MIN_CROSS_SECTION,
        tradability_mode='basic', missing_return_policy='zero',
        max_missing_return_ratio=MAX_MISSING_RETURN_RATIO,
        mad_n=5.0, standardize=False, neutralize=False,
        market_exchanges=EXCHANGES, save_holdings=False,
        allow_exit_after_end=False,
    )
    result = StrictFactorBacktester(paths, factor, config).run(save=True)
    print('回测输出：', backtest_output)
    display(result.ic_summary)
    display(result.performance.loc[[
        'Q1', 'Q10', 'long_short_gross', 'long_short_net'
    ]])
    display(result.audit[['fill_ratio', 'missing_return_ratio']].describe())
else:
    print('RUN_BACKTEST=False：因子已生成，回测请使用通用模板。')


## 7. 结果解释注意事项

- `amihud_20_raw` 是每亿元成交额对应的平均绝对价格变化；
- `amihud_20_factor` 是去极值、规模中性化后的最终信号；
- 高 Amihud 股票天然交易容量较差，统计收益不能直接视为可实现收益；
- 2015年可能出现大面积停牌，退出日没有VWAP的记录应在回测审计中重点检查；
- 当前没有行业字段，因此只做规模中性化；
- 2004–2005年只用于预热，正式因子严格输出2006-01-04至2017-12-29；
- 本 Notebook 不读取2018年以后的验证期和测试期数据。
